# Retrieval vs Planning Test — Qwen3.6-35B-A3B on SuperGPQA

**Critical question**: does the model *plan forward* on MCQ reasoning, or does it *retrieve* via letter-position?

Our prior results:
- Detection: probe L17 AUROC 0.78 at T=10 (+37.8pp over BOW) — signal exists
- V1/V2/factorial/SAE/transcoder intervention: null across all 5 methods
- Anomaly: ablation INCREASES source-kept by +15pp (amnesic, Elazar 2021)
- S4-L0 mapping: Cohen's d peak L22, logit-lens commit L39 — **17-layer gap**

**Hypothesis**: The model doesn't plan. It retrieves the answer via pattern-matching at L22, then binds to a letter position late (L39). The detection probe captures the binding readout, not an upstream plan.

**Test**: three behavioral signatures distinguish retrieval from reasoning. If any flips, the paper reframes.

| Test | Retrieval signature | Reasoning signature |
|---|---|---|
| Letter permutation | Answer follows letter (original position) | Answer follows content (new letter) |
| Option content swap | Answer stays on original letter | Answer follows swapped content |
| Thinking ON vs OFF | Small accuracy delta | Big accuracy delta |

Outcome decides paper framing:
- **Retrieval confirmed** → paper reframes from "forward planning" to **"semantic-to-letter binding circuit in hybrid MoE"**. All null interventions become consistent (no plan to intervene on).
- **Reasoning confirmed** → proceeds to multi-layer simultaneous patching (Etapa 2) as the actual causal test.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/retrieval_test')
OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load model + Stage B rollouts

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load Stage B for questions + correct rollouts
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
for shard in tqdm(shards, desc='load questions'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q_options = json.loads(meta['options'])
        rollouts = json.loads(meta['rollouts'])
        # Only questions where at least one rollout was correct (we'll use these)
        if not any(r['correct'] for r in rollouts): continue
        questions.append(dict(
            question=meta['question'],
            options=q_options,
            gold_letter=meta['gold_letter'],
            n_options=len(q_options),
        ))
print(f'{len(questions)} questions with at least 1 correct rollout')

## Cell 3 — Prompt helpers + letter permutation + content swap

Three prompt variants:
- **baseline**: original options, original letters
- **permuted**: options shuffled (same content, different letters)
- **thinking_off**: same prompt but explicit instruction for direct answer, no reasoning

In [ ]:
BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
def extract_letter(c, n_opts):
    m = list(BOXED_RE.finditer(c))
    if m:
        l = m[-1].group(1).upper()
        if ord(l)-ord('A') < n_opts: return l
    tail = c[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands and ord(cands[-1])-ord('A') < n_opts: return cands[-1]
    return None

def format_prompt(question, options, thinking=True):
    """Format MCQ prompt. If thinking=False, explicitly instruct for direct answer."""
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    if thinking:
        content = (
            "Answer the following multiple-choice question. Reason step by step, "
            "then put the letter of the correct answer in \\boxed{}.\n\n"
            f"Question: {question}\n\nOptions:\n{choices}")
    else:
        content = (
            "Answer the following multiple-choice question. "
            "Respond with ONLY the letter in \\boxed{}, no reasoning, no explanation.\n\n"
            f"Question: {question}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

def permute_options(options, gold_letter, seed):
    """Permute options and return (new_options, new_gold_letter, mapping).
    mapping[new_letter] = original_letter it came from."""
    n = len(options)
    rng = random.Random(seed)
    perm = list(range(n))
    rng.shuffle(perm)
    # Ensure it's actually a non-trivial permutation
    while perm == list(range(n)) and n > 1:
        rng.shuffle(perm)
    new_options = [options[i] for i in perm]
    # Find where gold ended up: gold was at idx = ord(gold)-65. It's now at new position j where perm[j] = idx.
    gold_idx = ord(gold_letter) - 65
    new_gold_idx = perm.index(gold_idx)
    new_gold_letter = chr(65 + new_gold_idx)
    # mapping: new_letter → original_letter
    mapping = {chr(65+j): chr(65+perm[j]) for j in range(n)}
    return new_options, new_gold_letter, mapping

def generate(prompt, max_new=1024):
    enc = tok(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids=enc.input_ids, attention_mask=enc.attention_mask,
            max_new_tokens=max_new, do_sample=False,
            pad_token_id=tok.pad_token_id, use_cache=True,
        )
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)

# Sanity check: permutation logic
q0 = questions[0]
new_opts, new_gold, mapping = permute_options(q0['options'], q0['gold_letter'], seed=42)
print(f'Original gold: {q0["gold_letter"]} = {q0["options"][ord(q0["gold_letter"])-65][:50]}')
print(f'Permuted gold: {new_gold} = {new_opts[ord(new_gold)-65][:50]}')
print(f'Mapping: {mapping}')
assert q0['options'][ord(q0['gold_letter'])-65] == new_opts[ord(new_gold)-65], 'permutation bug'
print('✅ permutation logic OK')

## Cell 4 — Run test suite (N=50, 3 conditions = 150 gens, ~2h)

For each of 50 questions:
- **baseline**: original prompt, thinking ON
- **permuted**: options shuffled, thinking ON (compare with baseline: does answer follow content or letter?)
- **thinking_off**: original prompt, thinking OFF (direct answer only)

Compute per rollout:
- `baseline_letter` — model's prediction on original
- `baseline_correct` — baseline_letter == gold_letter  
- `permuted_letter` — model's prediction on shuffled
- `permuted_correct` — permuted_letter == new_gold_letter (tracks content)
- `permuted_follows_original_letter` — permuted_letter == baseline_letter (tracks position/letter)
- `thinking_off_letter` — model's prediction with thinking OFF
- `thinking_off_correct` — direct answer accuracy

In [ ]:
N_TEST = 50
random.seed(42)
sample = random.sample(questions, N_TEST)

results = []
t0 = time.time()
for i, q in enumerate(tqdm(sample, desc='retrieval test')):
    try:
        # Baseline
        p_base = format_prompt(q['question'], q['options'], thinking=True)
        t_base = generate(p_base, max_new=1024)
        l_base = extract_letter(t_base, q['n_options'])

        # Permuted options
        new_opts, new_gold, mapping = permute_options(q['options'], q['gold_letter'], seed=1000+i)
        p_perm = format_prompt(q['question'], new_opts, thinking=True)
        t_perm = generate(p_perm, max_new=1024)
        l_perm = extract_letter(t_perm, q['n_options'])

        # Thinking OFF
        p_off = format_prompt(q['question'], q['options'], thinking=False)
        t_off = generate(p_off, max_new=64)  # short gen for direct answer
        l_off = extract_letter(t_off, q['n_options'])

        row = dict(
            idx=i,
            gold=q['gold_letter'],
            new_gold=new_gold,
            mapping=mapping,
            baseline_letter=l_base,
            permuted_letter=l_perm,
            thinking_off_letter=l_off,
            baseline_correct=(l_base == q['gold_letter']),
            permuted_correct=(l_perm == new_gold),
            thinking_off_correct=(l_off == q['gold_letter']),
            # key for retrieval vs planning: on permuted, which did the model pick?
            permuted_original_letter=mapping.get(l_perm) if l_perm else None,  # what letter was it originally?
            # If permuted_original_letter == baseline_letter, the model picked the SAME CONTENT as baseline (reasoning)
            # If permuted_letter == baseline_letter, the model picked the SAME LETTER regardless of content (retrieval)
        )
        results.append(row)
        if (i+1) % 5 == 0:
            baseline_acc = sum(r['baseline_correct'] for r in results)/len(results)
            perm_acc = sum(r['permuted_correct'] for r in results)/len(results)
            off_acc = sum(r['thinking_off_correct'] for r in results)/len(results)
            print(f'  [{i+1}/{N_TEST}] baseline acc={baseline_acc:.0%} | permuted acc={perm_acc:.0%} | thinking_off acc={off_acc:.0%}')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f'  OOM on rollout {i}, skipping')
        continue

with open(OUT/'retrieval_results.json', 'w') as f:
    json.dump(dict(n=len(results), results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ done in {(time.time()-t0)/60:.1f} min')

## Cell 5 — Analysis + verdict

In [ ]:
# Filter to only rollouts where all three gens were valid
valid = [r for r in results if r['baseline_letter'] and r['permuted_letter'] and r['thinking_off_letter']]
print(f'Valid rollouts (all 3 letters extracted): {len(valid)}/{len(results)}')
# And where baseline was correct (we want to test whether the CORRECT prediction tracks content or letter)
correct_base = [r for r in valid if r['baseline_correct']]
print(f'Correct on baseline: {len(correct_base)}')

print('\n=== Accuracy summary ===')
print(f'baseline         : {sum(r["baseline_correct"] for r in valid)/len(valid)*100:.1f}%')
print(f'permuted options : {sum(r["permuted_correct"] for r in valid)/len(valid)*100:.1f}%')
print(f'thinking OFF     : {sum(r["thinking_off_correct"] for r in valid)/len(valid)*100:.1f}%')

# CRITICAL: on permuted, does the model's answer track CONTENT or LETTER?
# "tracks content" = permuted_original_letter == baseline_letter (model picked same option, just different letter)
# "tracks letter" = permuted_letter == baseline_letter (model picked same LETTER regardless of content)
print('\n=== Letter-permutation signature (on correct-baseline rollouts) ===')
n_tracks_content = sum(1 for r in correct_base if r['permuted_original_letter'] == r['baseline_letter'])
n_tracks_letter = sum(1 for r in correct_base if r['permuted_letter'] == r['baseline_letter'])
n_neither = len(correct_base) - n_tracks_content - n_tracks_letter
print(f'tracks content (REASONING): {n_tracks_content}/{len(correct_base)} ({n_tracks_content/len(correct_base)*100:.1f}%)')
print(f'tracks letter (RETRIEVAL):  {n_tracks_letter}/{len(correct_base)} ({n_tracks_letter/len(correct_base)*100:.1f}%)')
print(f'neither (drift):           {n_neither}/{len(correct_base)} ({n_neither/len(correct_base)*100:.1f}%)')

# Thinking OFF delta
thinking_delta = (sum(r['baseline_correct'] for r in valid) - sum(r['thinking_off_correct'] for r in valid))/len(valid)*100
print(f'\n=== Thinking delta (reasoning vs retrieval) ===')
print(f'Δ accuracy (thinking ON − thinking OFF) = {thinking_delta:+.1f}pp')

# Verdict
print('\n=== VERDICT ===')
retrieval_score = 0  # higher = more retrieval-like
if len(correct_base) > 0:
    tracks_content_rate = n_tracks_content / len(correct_base)
    tracks_letter_rate = n_tracks_letter / len(correct_base)
    if tracks_letter_rate > tracks_content_rate:
        retrieval_score += 2
        print(f'❌ Letter-permutation FAIL: model tracks letter ({tracks_letter_rate*100:.0f}%) > content ({tracks_content_rate*100:.0f}%) → retrieval')
    elif tracks_content_rate > 0.7:
        retrieval_score -= 2
        print(f'✅ Letter-permutation PASS: model tracks content ({tracks_content_rate*100:.0f}%) → reasoning')
    else:
        retrieval_score += 0
        print(f'🤔 Letter-permutation MIXED: content {tracks_content_rate*100:.0f}% vs letter {tracks_letter_rate*100:.0f}% → ambiguous')

if abs(thinking_delta) < 5:
    retrieval_score += 1
    print(f'❌ Thinking delta small ({thinking_delta:+.1f}pp) → model performs similarly WITHOUT thinking → retrieval')
elif thinking_delta > 10:
    retrieval_score -= 1
    print(f'✅ Thinking delta large ({thinking_delta:+.1f}pp) → reasoning helps → reasoning')

print(f'\n=== FINAL DIAGNOSIS (retrieval_score={retrieval_score}) ===')
if retrieval_score >= 2:
    print('🔍 RETRIEVAL-DOMINATED — paper reframes to "semantic-to-letter binding circuit"')
    print('   All null interventions are now CONSISTENT: there is no upstream plan, so nothing to intervene on.')
    print('   Next experiment: map the binding circuit (L22 peak → L39 commit)')
elif retrieval_score <= -2:
    print('🧠 REASONING-DOMINATED — paper keeps forward-planning thesis')
    print('   Null interventions indicate single-layer insufficient. Next: multi-layer constrained patching.')
else:
    print('⚖️  MIXED — paper frames as "retrieval + some reasoning" with careful bounds')
    print('   Next: stratify by discipline / difficulty to find where reasoning actually helps.')